In [ ]:
import pandas as pd
import re
import os

FILE_PATH = r"c:\Users\hp\Documents\ds_materials\data_cleaning\data-for-calling_compress.xlsx"
OUTPUT_CSV = r"c:\Users\hp\Documents\ds_materials\data_cleaning\extracted_contacts.csv"

In [ ]:

# ── Step 1: Inspect the raw file ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1 – Inspecting raw Excel file")
print("=" * 60)

xl = pd.ExcelFile(FILE_PATH)
print(f"Sheet names: {xl.sheet_names}\n")


In [ ]:
for sheet in xl.sheet_names:
    df_raw = pd.read_excel(FILE_PATH, sheet_name=sheet, header=None)
    print(f"--- Sheet: '{sheet}' | Shape: {df_raw.shape} ---")
    print(df_raw.head().to_string())
    print()


In [ ]:
df_raw = pd.read_excel(FILE_PATH, sheet_name=sheet, header=None)
df_raw.isnull().sum()

In [ ]:
len(df_raw)

In [ ]:
df_raw.head()

In [ ]:
import pandas as pd

# Sample: df = pd.read_csv("your_file.csv")

new_data = []

for _, row in df_raw.iterrows():
    # Get non-null values in the row
    values = row.dropna().values
    
    # Append to list
    new_data.append(values)

# Convert to DataFrame
new_df = pd.DataFrame(new_data)

new_df.to_csv('new.csv')

In [ ]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("new.csv")

final_data = []

for _, row in df.iterrows():
    
    # Extract columns
    s_no = str(row[0])
    names = str(row[1])
    phones = str(row[2])
    address = str(row[3])
    
    # Skip empty rows
    if s_no == 'nan' and names == 'nan':
        continue
    
    # Split by newline
    s_no_list = [x.strip() for x in s_no.split("\n") if x.strip() and x.strip().isdigit()]
    name_list = [x.strip() for x in names.split("\n") if x.strip()]
    phone_list = [x.strip() for x in phones.split("\n") if x.strip()]
    addr_list = [x.strip() for x in address.split("\n") if x.strip()]
    
    # Find max length
    max_len = max(len(s_no_list), len(name_list), len(phone_list), len(addr_list))
    
    # Pad lists
    s_no_list += [None] * (max_len - len(s_no_list))
    name_list += [None] * (max_len - len(name_list))
    phone_list += [None] * (max_len - len(phone_list))
    addr_list += [None] * (max_len - len(addr_list))
    
    # Combine row-wise
    for i in range(max_len):
        final_data.append([
            s_no_list[i],
            name_list[i],
            phone_list[i],
            addr_list[i]
        ])

# Create final DataFrame
clean_df = pd.DataFrame(final_data, columns=[
    "serial_number", "name", "phone_number", "address"
])

# Remove rows where everything is null
clean_df = clean_df.dropna(how='all')

# Optional: clean phone numbers
clean_df["phone_number"] = clean_df["phone_number"].str.extract(r'(\d{10})')

print(clean_df.head())

In [ ]:
clean_df.to_csv('new2.csv')

In [ ]:
import pandas as pd
import re

df = pd.read_csv("new.csv")

final_rows = []

for _, row in df.iterrows():
    
    s_no_text = str(row[0])
    name_text = str(row[1])
    phone_text = str(row[2])
    addr_text = str(row[3])
    
    # Skip empty rows
    if s_no_text == 'nan' and name_text == 'nan':
        continue
    
    # ---------------------------
    # 1. Extract Serial + Names
    # ---------------------------
    combined = s_no_text + "\n" + name_text
    
    # pattern: number followed by name
    matches = re.findall(r'(\d+)\s*\n?([A-Za-z ]+)', combined)
    
    serials = []
    names = []
    
    for s, n in matches:
        serials.append(s.strip())
        names.append(n.strip())
    
    # ---------------------------
    # 2. Extract Phones
    # ---------------------------
    phones = re.findall(r'\d{10}', phone_text)
    
    # ---------------------------
    # 3. Extract Address Blocks
    # ---------------------------
    addr_lines = [x.strip() for x in addr_text.split("\n") if x.strip()]
    
    # Merge address lines smartly
    addresses = []
    temp = ""
    
    for line in addr_lines:
        # New address starts when line contains number + comma or H.No etc
        if re.search(r'\d|H\.No|Plot|Sector|House|Flat', line) and temp:
            addresses.append(temp.strip())
            temp = line
        else:
            temp += " " + line
    
    if temp:
        addresses.append(temp.strip())
    
    # ---------------------------
    # 4. Align Data
    # ---------------------------
    max_len = max(len(serials), len(names), len(phones), len(addresses))
    
    for i in range(max_len):
        final_rows.append([
            serials[i] if i < len(serials) else None,
            names[i] if i < len(names) else None,
            phones[i] if i < len(phones) else None,
            addresses[i] if i < len(addresses) else None
        ])

# ---------------------------
# Final DataFrame
# ---------------------------
clean_df = pd.DataFrame(final_rows, columns=[
    "serial_number", "name", "phone_number", "address"
])

# Remove bad rows
clean_df = clean_df.dropna(subset=["serial_number", "name"])

clean_df.head(10)

In [23]:
import pandas as pd
import re

df = pd.read_csv("new.csv")

final_data = []

for _, row in df.iterrows():
    
    s_no_text = str(row[0])
    name_text = str(row[1])
    phone_text = str(row[2])
    addr_text = str(row[3])

    # ----------------------------
    # 1. Extract Serial Numbers
    # ----------------------------
    serials = re.findall(r'\b\d+\b', s_no_text)

    # ----------------------------
    # 2. Extract Phone Numbers (KEY)
    # ----------------------------
    phones = re.findall(r'\d{10}', phone_text)

    # ----------------------------
    # 3. Extract Names (split smartly)
    # ----------------------------
    # Split by newline first
    raw_names = []
    for line in name_text.split("\n"):
        parts = re.findall(r'[A-Z][a-z]+(?:\s?[A-Z][a-z]+)*', line)
        raw_names.extend(parts)

    # ----------------------------
    # 4. Extract Addresses (group lines)
    # ----------------------------
    addr_lines = [x.strip() for x in addr_text.split("\n") if x.strip()]

    addresses = []
    temp = ""

    for line in addr_lines:
        # Start new address if strong indicator
        if re.search(r'(H\.No|Plot|Sector|House|Flat|\d{2,})', line) and temp:
            addresses.append(temp.strip())
            temp = line
        else:
            temp += " " + line

    if temp:
        addresses.append(temp.strip())

    # ----------------------------
    # 5. Align using PHONE COUNT
    # ----------------------------
    n = len(phones)

    # Fix lengths
    serials = (serials + [None]*n)[:n]
    raw_names = (raw_names + [None]*n)[:n]
    addresses = (addresses + [None]*n)[:n]

    # ----------------------------
    # 6. Build Final Rows
    # ----------------------------
    for i in range(n):
        final_data.append([
            serials[i],
            raw_names[i],
            phones[i],
            addresses[i]
        ])

# ----------------------------
# Final DataFrame
# ----------------------------
clean_df = pd.DataFrame(final_data, columns=[
    "serial_number", "name", "phone_number", "address"
])

# Remove invalid rows
clean_df = clean_df.dropna(subset=["phone_number"])

clean_df.head(10)

C:\Users\hp\AppData\Local\Temp\ipykernel_31312\2196198182.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  s_no_text = str(row[0])
C:\Users\hp\AppData\Local\Temp\ipykernel_31312\2196198182.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  name_text = str(row[1])
C:\Users\hp\AppData\Local\Temp\ipykernel_31312\2196198182.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  phone_text = str(row[2])
C:\Users\hp\AppData\Local\Temp

,serial_number,name,phone_number,address
0,2,Deepakchoudhary,9414712244,barso teh+diss.-bharatpur state-rajasthan MARK...
1,None,Bhuvneshmittal,9828785369,"1-C/40, Shiv Shakti Colony, Shastri Nagar, Jaipur"
2,None,Manoj Chawla,9799926575,A-446 Vaishali Nagar
3,None,Sanjay KumarDubey,9799393355,"Jaipur Rajasthan 302021 104(A) KAILASH NAGAR ,..."
4,None,VireshJhalanee,9829096803,Plot no.81 kanak vihar colony Agra road jaipur...
5,None,Pankaj,9828262454,"MEERUT CANTT, MEERUT-250001 (U.P.)."
6,None,RameshYadav,9829464356,"Plot NO. D-255, Sector No.-8, Vidhy dhar Nagar..."
7,None,Mahendra PrasadKhandelwal,9929222077,"kanwar nagar jaipur 302002 Shivaji Marg, Nehru"
8,None,Sanjaythakur,9314142257,"Nagar, Jaipur, Rajasthan - 302016"
9,None,Raees AhmedAnsari,9983952230,"6/381, Opp. Shiv Mandir, New Vidhyadhar Nagar,..."
